# OEC FLUX.2 Klein — Gradio client for the model server

**UI environment:** `phase2_3d` (requests + Gradio frontend). FLUX inference runs in the separate `flux-api` sidecar; this notebook deliberately does not import PyTorch or Diffusers and does not install packages into the shared kernel.

Start the model server from the OEC repository root on an idle A100:

```bash
python tools/oec_service_manager.py start flux-api --gpu <ID> --port 7867 --allow-download
python tools/oec_service_manager.py status flux-api
python tools/oec_service_manager.py logs flux-api --tail 80
```

The sidecar verifies the pinned FLUX dependency stack, uses StorageManager admission for any required cache fill, checks model readiness, and writes generated PNG assets under OEC-managed outputs. Use `--allow-download` only when a cache fill is needed and permitted. This Gradio frontend binds to loopback; use an SSH tunnel for remote access. The share option is disabled.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import time
import urllib.parse
import gradio as gr
import requests
from PIL import Image

API_BASE = os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7867").rstrip("/")
RESULT_ROOT = Path("/tmp") / "oec-notebook-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if urllib.parse.urlsplit(API_BASE).hostname not in {"127.0.0.1", "localhost", "::1"}:
    raise RuntimeError("FLUX notebook only permits a loopback OEC media API endpoint.")
HTTP = requests.Session()
HTTP.trust_env = False

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def service_health():
    try:
        response = HTTP.get(f"{API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(
            f"FLUX model server is unreachable at {API_BASE}. Start the approved flux-api service first."
        ) from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"FLUX model server is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

def submit_and_wait(payload, timeout_seconds=1800):
    health = service_health()
    response = HTTP.post(f"{API_BASE}/v1/jobs", json=payload, timeout=30)
    if response.status_code != 202:
        raise gr.Error(f"FLUX job submission refused (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected FLUX job response: {accepted}")
    deadline = time.monotonic() + timeout_seconds
    latest = accepted
    while time.monotonic() < deadline:
        poll = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"FLUX job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"FLUX job {status}: {latest.get('message', 'No error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error("FLUX generation timed out; inspect the flux-api service logs.")
    result = HTTP.get(f"{API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"FLUX result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "").split(";", 1)[0].lower()
    if content_type != "image/png":
        raise gr.Error(f"Expected image/png from FLUX, received {content_type!r}.")
    output_path = RESULT_ROOT / f"{job_id}.png"
    if output_path.exists():
        raise gr.Error("Refusing to overwrite an existing notebook result.")
    output_path.write_bytes(result.content)
    if output_path.stat().st_size <= 64 or not output_path.read_bytes().startswith(b"\x89PNG\r\n\x1a\n"):
        output_path.unlink(missing_ok=True)
        raise gr.Error("FLUX server returned an empty or invalid PNG.")
    try:
        with Image.open(output_path) as check:
            check.verify()
        with Image.open(output_path) as check:
            width, height = check.size
            if width <= 0 or height <= 0:
                raise ValueError("image dimensions are empty")
    except Exception as exc:
        output_path.unlink(missing_ok=True)
        raise gr.Error("FLUX result is a PNG signature but cannot be decoded as an image.") from exc
    details = {
        "job_id": job_id, "status": latest.get("status"), "api": API_BASE,
        "health": health, "metadata": latest.get("metadata", {}),
        "artifact_path": str(output_path), "artifact_bytes": output_path.stat().st_size,
        "artifact_sha256": sha256_bytes(output_path.read_bytes()), "width": width, "height": height,
    }
    return str(output_path), json.dumps(details, indent=2, ensure_ascii=False)

In [ ]:
def generate_flux(prompt, seed, width, height, steps, guidance_scale):
    if not isinstance(prompt, str) or not prompt.strip():
        raise gr.Error("Enter an image prompt.")
    seed, width, height, steps = int(seed), int(width), int(height), int(steps)
    if not 0 <= seed <= 2**31 - 1:
        raise gr.Error("Seed must be between 0 and 2147483647.")
    if width not in {512, 768, 1024} or height not in {512, 768, 1024}:
        raise gr.Error("Width and height must be 512, 768, or 1024.")
    if not 1 <= steps <= 20 or not 0 <= float(guidance_scale) <= 10:
        raise gr.Error("Steps or guidance scale is outside the supported range.")
    payload = {
        "task": "text_to_image", "prompt": prompt.strip(),
        "parameters": {"seed": seed, "width": width, "height": height,
                       "steps": steps, "guidance_scale": float(guidance_scale)},
    }
    return submit_and_wait(payload)

demo = gr.Interface(
    fn=generate_flux,
    inputs=[
        gr.Textbox(label="Image prompt", lines=3, value="A photorealistic amber glass sphere on a clean studio background"),
        gr.Number(value=42, precision=0, label="Seed"),
        gr.Dropdown([512, 768, 1024], value=512, label="Width"),
        gr.Dropdown([512, 768, 1024], value=512, label="Height"),
        gr.Slider(1, 20, value=4, step=1, label="Inference steps"),
        gr.Slider(0, 10, value=1.0, step=0.1, label="Guidance scale"),
    ],
    outputs=[gr.Image(type="filepath", label="Generated PNG"), gr.Textbox(label="Server and artifact metadata", lines=12)],
    title="OEC FLUX.2 Klein — model server",
    description="Inference is executed by the separate FLUX provider API. The callback downloads and validates the resulting PNG.",
)
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(
        server_name="127.0.0.1", server_port=7904, share=False, inline=False, show_error=True, allowed_paths=[str(RESULT_ROOT)]
    )